In [1]:
from IPython.core.display import HTML
table_css = 'table {align:left;display:block} '
HTML('<style>{}</style>'.format(table_css))

# 🎯 Exercise 1: Forward Mode Automatic Differentiation for TinyML
## MAIE 5532: Machine Learning Systems - Week 2

### Learning Objectives:
- Understand dual number representation for forward mode AD
- Implement forward mode AD for sensor data processing
- Validate AD results using numerical differentiation
- Connect theory to practical TinyML applications

### 🧠 What is Forward Mode Automatic Differentiation?

**Forward Mode AD** is like having a mathematical assistant that computes derivatives alongside your original calculation. Instead of just computing f(x), it simultaneously computes both f(x) AND f'(x) in one pass through your computation.

**Why is this revolutionary?**
- **Exact derivatives:** No approximation errors like finite differences
- **Efficient:** Only about twice the computational cost of the original function
- **Automatic:** No need to manually derive complex derivative formulas
- **Perfect for TinyML:** Predictable memory usage, ideal for embedded systems

### Real-World Impact:
Every time you use machine learning - from voice assistants to recommendation systems - automatic differentiation is working behind the scenes, computing the gradients that make learning possible.

## 📚 Mathematical Foundation: Dual Numbers

### What are Dual Numbers?
Think of dual numbers as "enhanced" numbers that carry two pieces of information:

**Dual Number = (value, derivative)**

Written mathematically as: *a + b·ε* where ε² = 0

### Real-world analogy: 
Imagine you're tracking both your current position AND your velocity while driving. The position is your "value" and velocity is your "derivative" (rate of change).

### Dual Number Arithmetic Rules:
- **Addition:** (a₁, b₁) + (a₂, b₂) = (a₁ + a₂, b₁ + b₂)
- **Multiplication:** (a₁, b₁) × (a₂, b₂) = (a₁ × a₂, a₁ × b₂ + b₁ × a₂)
- **Functions:** sin(a, b) = (sin(a), cos(a) × b)

### Key Insight: 
These rules automatically implement the chain rule of calculus! When we multiply two dual numbers, we're automatically applying the product rule: (f × g)' = f' × g + f × g'

### Why This Matters for Machine Learning:
Instead of manually deriving complex derivative formulas (which becomes impossible for neural networks with millions of parameters), we let the computer automatically apply these simple rules to compute exact derivatives.

In [2]:
# Essential imports for our forward mode AD implementation
import math      # For mathematical functions like sin, cos
import numpy as np  # For numerical operations (used in verification)

# Utility function for clean output display
def show(title, *pairs):
    """
    Pretty printer for displaying results
    
    This function makes our output more readable by formatting it nicely.
    
    Args:
        title: Description of what we're showing
        *pairs: Tuples of (label, value) to display
        
    The *pairs syntax means "accept any number of arguments"
    This lets us call: show("title", ("label1", value1), ("label2", value2), ...)
    """
    print(title)
    for k, v in pairs:
        print(f"  {k}: {v}")

print("✅ Imports completed successfully!")
print()
print("📝 Code Explanation:")
print("   • math: Provides sin(), cos() functions that work with regular numbers")
print("   • numpy: Used later for numerical gradient checking")
print("   • show() function: Makes our output readable - think of it as a formatted print statement")
print()
print("🐍 Python Concept - *args:")
print("   The *pairs means 'accept any number of arguments'")
print("   This is Python's way of making functions flexible")
print("   Like a restaurant menu that says 'pick any toppings you want'")

✅ Imports completed successfully!

📝 Code Explanation:
   • math: Provides sin(), cos() functions that work with regular numbers
   • numpy: Used later for numerical gradient checking
   • show() function: Makes our output readable - think of it as a formatted print statement

🐍 Python Concept - *args:
   The *pairs means 'accept any number of arguments'
   This is Python's way of making functions flexible
   Like a restaurant menu that says 'pick any toppings you want'


### 🔧 Understanding Python Concepts Used:

#### Python Magic Methods:
Python has special methods that start and end with double underscores (__). These are called "magic methods" or "dunder methods":

- **__init__:** Constructor - called when you create a new object
- **__add__:** Called when you use the + operator
- **__mul__:** Called when you use the * operator
- **__repr__:** Called when you print the object

#### isinstance() Function:
This checks if an object is of a specific type. Like asking "Is this a Dual number or a regular number?"

```python
isinstance(5, int)        # True - 5 is an integer
isinstance(5.0, float)    # True - 5.0 is a float
isinstance(x, Dual)       # True if x is a Dual number
```
#### Why This Matters:
Our Dual class needs to work with both Dual numbers AND regular numbers. The isinstance() function helps us handle both cases automatically.

## Code - Complete Dual Class:

In [3]:
class Dual:
    """
    Dual number class for forward mode automatic differentiation
    
    Think of this as a "smart number" that remembers both:
    - v: the actual value (like 2.5)
    - d: the derivative value (like 0.8)
    
    This is the heart of forward mode AD!
    """
    
    def __init__(self, v, d=0.0):
        """
        Create a new dual number - this is the constructor
        
        Args:
            v: The value (e.g., 2.5)
            d: The derivative (e.g., 1.0 for input variable, 0.0 for constants)
            
        Examples:
            x = Dual(2.0, 1.0)  # Input variable: x=2.0, dx/dx=1.0
            c = Dual(5.0, 0.0)  # Constant: c=5.0, dc/dx=0.0
        """
        self.v = float(v)  # Convert to float to ensure proper arithmetic
        self.d = float(d)  # Convert to float to ensure proper arithmetic
        print(f"Created dual number: value={self.v}, derivative={self.d}")
    
    def _wrap(self, other):
        """
        Helper function: convert regular numbers to Dual numbers
        
        This is like having an automatic translator:
        - If 'other' is already a Dual number → return it as-is
        - If 'other' is a regular number → convert it to Dual(other, 0.0)
        
        Why derivative = 0.0 for regular numbers?
        Because constants have zero derivative!
        Example: d/dx(5) = 0
        """
        if isinstance(other, Dual):
            return other
        else:
            print(f"Converting regular number {other} to Dual({other}, 0.0)")
            return Dual(other, 0.0)
    
    def __add__(self, other):
        """
        Addition of dual numbers: (a,b) + (c,d) = (a+c, b+d)
        
        Mathematical rule: (f + g)' = f' + g'
        
        This implements the calculus rule that the derivative of a sum
        is the sum of the derivatives.
        """
        o = self._wrap(other)
        result = Dual(self.v + o.v, self.d + o.d)
        print(f"Addition: {self} + {o} = {result}")
        print(f"  Value: {self.v} + {o.v} = {result.v}")
        print(f"  Derivative: {self.d} + {o.d} = {result.d}")
        return result
    
    def __radd__(self, other):
        """
        Reverse addition: handles cases like 5 + dual_number
        
        When Python sees: 5 + dual_number
        It first tries: (5).__add__(dual_number) 
        Since regular numbers don't know about Dual, this fails
        Then Python tries: dual_number.__radd__(5)
        That's when this method is called!
        """
        return self.__add__(other)
    
    def __sub__(self, other):
        """
        Subtraction: (a,b) - (c,d) = (a-c, b-d)
        
        Mathematical rule: (f - g)' = f' - g'
        """
        o = self._wrap(other)
        result = Dual(self.v - o.v, self.d - o.d)
        print(f"Subtraction: {self} - {o} = {result}")
        return result
    
    def __rsub__(self, other):
        """Reverse subtraction: handles cases like 5 - dual_number"""
        o = self._wrap(other)
        result = Dual(o.v - self.v, o.d - self.d)
        print(f"Reverse subtraction: {other} - {self} = {result}")
        return result
    
    def __mul__(self, other):
        """
        Multiplication: (a,b) * (c,d) = (a*c, a*d + b*c)
        
        This is the PRODUCT RULE from calculus!
        Mathematical rule: (f * g)' = f' * g + f * g'
        
        Why this formula?
        If f(x) has value 'a' and derivative 'b'
        And g(x) has value 'c' and derivative 'd'
        Then (f × g)(x) = a × c (value part)
        And (f × g)'(x) = b × c + a × d (derivative part)
        """
        o = self._wrap(other)
        result_value = self.v * o.v
        result_derivative = self.d * o.v + self.v * o.d
        result = Dual(result_value, result_derivative)
        
        print(f"Multiplication: {self} * {o} = {result}")
        print(f"  Value: {self.v} * {o.v} = {result_value}")
        print(f"  Derivative (Product Rule): {self.d} * {o.v} + {self.v} * {o.d} = {result_derivative}")
        print(f"  This automatically implements: (f*g)' = f'*g + f*g'")
        
        return result
    
    def __rmul__(self, other):
        """Reverse multiplication: handles cases like 5 * dual_number"""
        return self.__mul__(other)
    
    def __truediv__(self, other):
        """
        Division: (a,b) / (c,d) = (a/c, (b*c - a*d)/(c²))
        
        This is the QUOTIENT RULE from calculus!
        Mathematical rule: (f/g)' = (f'*g - f*g')/g²
        """
        o = self._wrap(other)
        result_value = self.v / o.v
        result_derivative = (self.d * o.v - self.v * o.d) / (o.v * o.v)
        result = Dual(result_value, result_derivative)
        
        print(f"Division: {self} / {o} = {result}")
        print(f"  Value: {self.v} / {o.v} = {result_value}")
        print(f"  Derivative (Quotient Rule): ({self.d} * {o.v} - {self.v} * {o.d}) / {o.v}² = {result_derivative}")
        
        return result
    
    def __rtruediv__(self, other):
        """Reverse division: handles cases like 5 / dual_number"""
        o = self._wrap(other)
        result_value = o.v / self.v
        result_derivative = (o.d * self.v - o.v * self.d) / (self.v * self.v)
        result = Dual(result_value, result_derivative)
        print(f"Reverse division: {other} / {self} = {result}")
        return result
    
    def __repr__(self):
        """
        String representation for easy debugging
        
        This is called when you print a Dual number or display it in Jupyter
        """
        return f"Dual(value={self.v}, derivative={self.d})"

print("✅ Dual class implemented successfully!")
print()
print("🧮 What We Just Built:")
print("   • A 'smart number' that tracks both values AND derivatives")
print("   • Automatic implementation of calculus rules (product rule, quotient rule)")
print("   • The foundation for computing exact derivatives in machine learning")
print()
print("🎯 Key Insight:")
print("   Every time we do arithmetic with Dual numbers, we're automatically")
print("   applying the chain rule from calculus. This is the magic of AD!")

✅ Dual class implemented successfully!

🧮 What We Just Built:
   • A 'smart number' that tracks both values AND derivatives
   • Automatic implementation of calculus rules (product rule, quotient rule)
   • The foundation for computing exact derivatives in machine learning

🎯 Key Insight:
   Every time we do arithmetic with Dual numbers, we're automatically
   applying the chain rule from calculus. This is the magic of AD!


## 🔍 Deep Dive: Understanding What We Just Built

### The Magic of the Product Rule Implementation

When we multiply two dual numbers, something beautiful happens. Let's break down the `__mul__` method:

```python
def __mul__(self, other):
    result_value = self.v * o.v           # f(x) * g(x)
    result_derivative = self.d * o.v + self.v * o.d  # f'(x)*g(x) + f(x)*g'(x)
```

### This IS the product rule from calculus!

### Why This Is Revolutionary:
#### Before Automatic Differentiation:
- Manually derive derivative formulas (error-prone, time-consuming)
- Use finite differences (approximate, slow)
- Limited to simple functions

#### With Automatic Differentiation:
- Write your function naturally using +, -, *, /
- Get EXACT derivatives automatically
- Works for arbitrarily complex functions
- Scales to neural networks with millions of parameters

#### Real-World Example:
When TensorFlow computes gradients for a neural network with 175 billion parameters (like GPT-3), it's using these same basic principles - just applied millions of times automatically!

#### Memory Efficiency:
Notice that each Dual number only stores 2 floats (value + derivative). This is why forward mode AD is perfect for embedded systems - predictable, minimal memory usage.

## Code - Mathematical Functions:

In [4]:
def sin(x):
    """
    Sine function for dual numbers
    
    Mathematical rule: d/dx[sin(x)] = cos(x)
    
    This implements the CHAIN RULE automatically:
    If x = Dual(value, derivative), representing some function g(x) and g'(x)
    Then sin(x) = Dual(sin(value), cos(value) * derivative)
    
    This gives us sin(g(x)) and its derivative cos(g(x)) * g'(x)
    """
    if isinstance(x, Dual):
        result_value = math.sin(x.v)
        result_derivative = math.cos(x.v) * x.d
        result = Dual(result_value, result_derivative)
        
        print(f"sin({x}) = {result}")
        print(f"  Value: sin({x.v}) = {result_value}")
        print(f"  Derivative: cos({x.v}) * {x.d} = {result_derivative}")
        print(f"  Chain rule: d/dx[sin(g(x))] = cos(g(x)) * g'(x)")
        
        return result
    return math.sin(x)  # Handle regular numbers too

def cos(x):
    """
    Cosine function for dual numbers
    
    Mathematical rule: d/dx[cos(x)] = -sin(x)
    Note the negative sign!
    """
    if isinstance(x, Dual):
        result_value = math.cos(x.v)
        result_derivative = -math.sin(x.v) * x.d  # Note the negative sign!
        result = Dual(result_value, result_derivative)
        
        print(f"cos({x}) = {result}")
        print(f"  Value: cos({x.v}) = {result_value}")
        print(f"  Derivative: -sin({x.v}) * {x.d} = {result_derivative}")
        
        return result
    return math.cos(x)

def relu(x):
    """
    ReLU (Rectified Linear Unit) function for dual numbers
    
    ReLU(x) = max(0, x) = x if x > 0, else 0
    
    Mathematical rule for derivatives:
    - If x > 0: ReLU(x) = x,     so ReLU'(x) = 1
    - If x ≤ 0: ReLU(x) = 0,     so ReLU'(x) = 0
    
    This makes ReLU a "gradient gate":
    - When active (x > 0): passes gradients through unchanged
    - When inactive (x ≤ 0): blocks all gradients
    
    This is why ReLU is so popular in deep learning!
    """
    if isinstance(x, Dual):
        if x.v > 0:
            # Active case: pass through both value and derivative
            result = Dual(x.v, x.d)
            print(f"ReLU({x}) = {result} (ACTIVE - gradient flows)")
            print(f"  Since {x.v} > 0, ReLU passes value and derivative through")
        else:
            # Inactive case: zero out both value and derivative
            result = Dual(0.0, 0.0)
            print(f"ReLU({x}) = {result} (INACTIVE - gradient blocked)")
            print(f"  Since {x.v} ≤ 0, ReLU outputs 0 for both value and derivative")
        
        return result
    
    # Handle regular numbers
    return x if x > 0 else 0.0

print("✅ Mathematical functions implemented!")
print()
print("🧮 Chain Rule Implementation:")
print("   • sin(x): Automatically applies d/dx[sin(g(x))] = cos(g(x)) * g'(x)")
print("   • cos(x): Automatically applies d/dx[cos(g(x))] = -sin(g(x)) * g'(x)")
print("   • ReLU(x): Acts as a 'gradient gate' - crucial for deep learning!")
print()
print("🎯 Why This Matters:")
print("   These functions can now be used in any complex computation")
print("   and we'll automatically get exact derivatives!")

✅ Mathematical functions implemented!

🧮 Chain Rule Implementation:
   • sin(x): Automatically applies d/dx[sin(g(x))] = cos(g(x)) * g'(x)
   • cos(x): Automatically applies d/dx[cos(g(x))] = -sin(g(x)) * g'(x)
   • ReLU(x): Acts as a 'gradient gate' - crucial for deep learning!

🎯 Why This Matters:
   These functions can now be used in any complex computation
   and we'll automatically get exact derivatives!


## 📐 Mathematical Functions: Chain Rule in Action

### Understanding the Chain Rule Implementation

Each function implements the **chain rule** automatically:

**Chain Rule:** If y = f(g(x)), then dy/dx = f'(g(x)) × g'(x)

### For sin(x):
- Input: Dual(value, derivative) representing g(x) and g'(x)
- Output: Dual(sin(value), cos(value) × derivative)
- This gives us sin(g(x)) and its derivative cos(g(x)) × g'(x)

### Understanding ReLU's Special Role

ReLU is particularly important in deep learning because:

1. **Computational Efficiency:** Simple max(0, x) operation
2. **Gradient Properties:** Either passes gradients (1) or blocks them (0)
3. **Solves Vanishing Gradients:** Unlike sigmoid/tanh, ReLU doesn't saturate for positive values

### Why ReLU Derivatives Matter:
- **Active neurons (x > 0):** Gradient = 1, learning continues
- **Inactive neurons (x ≤ 0):** Gradient = 0, neuron stays "off"

This creates sparse activation patterns that help neural networks learn efficiently.

### Real-World Impact:
The introduction of ReLU revolutionized deep learning. Before ReLU, training deep networks was extremely difficult due to vanishing gradients. ReLU's simple derivative properties (0 or 1) solved this problem.

In [5]:
# Let's see our Dual numbers in action with a simple example
print("=" * 60)
print("🎯 SIMPLE EXAMPLE: f(x) = x² × sin(x) at x = 2.0")
print("=" * 60)
print()

print("This example shows how forward mode AD works step by step.")
print("We'll compute both the function value AND its derivative simultaneously.")
print()

# Step 1: Create input variable
print("STEP 1: Create input variable")
print("-" * 30)
x = Dual(2.0, 1.0)  # x = 2.0, dx/dx = 1.0
print(f"Created input: {x}")
print(f"Interpretation: x = {x.v}, and since x is our input variable, dx/dx = {x.d}")
print()

# Step 2: Compute x^2
print("STEP 2: Compute x²")
print("-" * 30)
print("We'll use our multiplication operator which implements the product rule:")
a = x * x
print()

# Step 3: Compute sin(x)
print("STEP 3: Compute sin(x)")
print("-" * 30)
print("We'll use our sin function which implements the chain rule:")
b = sin(x)
print()

# Step 4: Compute final result
print("STEP 4: Compute x² × sin(x)")
print("-" * 30)
print("Final multiplication using the product rule:")
y = a * b
print()

print("=" * 60)
print("🎯 FINAL RESULT")
print("=" * 60)
show("f(x) = x² × sin(x) at x = 2.0",
     ("Function value f(2.0)", y.v),
     ("Derivative f'(2.0)", y.d))

print()
print("🔍 What This Means:")
print(f"   • The function value at x=2 is {y.v:.6f}")
print(f"   • The slope (derivative) at x=2 is {y.d:.6f}")
print(f"   • If we change x by a tiny amount δx, f(x) changes by approximately {y.d:.6f} × δx")
print()
print("🎉 We computed this derivative without any manual calculus!")
print("   The chain rule was applied automatically through our Dual number operations.")

🎯 SIMPLE EXAMPLE: f(x) = x² × sin(x) at x = 2.0

This example shows how forward mode AD works step by step.
We'll compute both the function value AND its derivative simultaneously.

STEP 1: Create input variable
------------------------------
Created dual number: value=2.0, derivative=1.0
Created input: Dual(value=2.0, derivative=1.0)
Interpretation: x = 2.0, and since x is our input variable, dx/dx = 1.0

STEP 2: Compute x²
------------------------------
We'll use our multiplication operator which implements the product rule:
Created dual number: value=4.0, derivative=4.0
Multiplication: Dual(value=2.0, derivative=1.0) * Dual(value=2.0, derivative=1.0) = Dual(value=4.0, derivative=4.0)
  Value: 2.0 * 2.0 = 4.0
  Derivative (Product Rule): 1.0 * 2.0 + 2.0 * 1.0 = 4.0
  This automatically implements: (f*g)' = f'*g + f*g'

STEP 3: Compute sin(x)
------------------------------
We'll use our sin function which implements the chain rule:
Created dual number: value=0.9092974268256817, deriva

## 🎯 Exercise 1: TinyML Sensor Data Processing

### Problem Scenario
**Context:** You're developing a TinyML system for environmental monitoring. The system processes sensor readings (temperature, humidity, light levels, etc.) before feeding them to a neural network for pattern recognition.

### The Processing Pipeline:
Your sensor preprocessing consists of three steps:

1. **Normalize:** Convert raw sensor reading from [0, 1024] to [-1, 1] range
   - Formula: `(sensor_value - 512) / 512`
   - Why? Neural networks work better with normalized inputs

2. **Activate:** Apply ReLU to remove negative values
   - Formula: `max(0, normalized_value)`
   - Why? Sometimes we only care about positive deviations

3. **Scale:** Multiply by 0.5 for the final feature
   - Formula: `activated_value * 0.5`
   - Why? Keeps values in a controlled range for the neural network

### Your Task:
Implement this processing function and compute its derivative using forward mode AD. 

### Why Do We Need the Derivative?

#### For On-Device Learning:
- **Federated Learning:** Your device needs to compute gradients locally
- **Continual Learning:** Adapt the preprocessing based on new data patterns
- **Transfer Learning:** Fine-tune preprocessing for specific environments

#### For Analysis:
- **Sensitivity Analysis:** How sensitive is our output to sensor noise?
- **Robustness:** Will small sensor errors cause large output changes?
- **Feature Engineering:** Should we adjust our preprocessing parameters?

#### For System Design:
- **Gradient Flow:** Ensuring gradients can flow back through preprocessing
- **Numerical Stability:** Avoiding regions where gradients explode or vanish
- **Hardware Optimization:** Understanding computational requirements

In [6]:
def tinyml_feature_processing(s):
    """
    TinyML sensor data processing pipeline with detailed step-by-step explanation
    
    This function represents a typical preprocessing pipeline in embedded ML systems.
    We'll process sensor data through normalization, activation, and scaling.
    
    Pipeline Steps:
    1. Normalize: (s - 512) / 512  [maps 0-1024 range to -1 to +1]
    2. Activate: ReLU(normalized)  [removes negative values]
    3. Scale: result * 0.5         [final scaling for neural network input]
    
    Args:
        s: Sensor reading (can be regular number or Dual number)
    
    Returns:
        Processed feature value (same type as input)
    """
    # Ensure input is Dual if we want derivatives
    s = s if isinstance(s, Dual) else Dual(float(s), 0.0)
    
    print("🔧 TINYML SENSOR PROCESSING PIPELINE")
    print("=" * 50)
    print(f"📊 Input sensor reading: {s}")
    print(f"   Raw sensor value: {s.v}")
    print(f"   Input derivative: {s.d} (1.0 means this is our input variable)")
    print()
    
    # Step 1: Normalize to [-1, 1] range
    print("STEP 1: NORMALIZATION")
    print("-" * 25)
    print("🎯 Goal: Convert sensor range [0, 1024] to [-1, 1]")
    print("📐 Formula: (s - 512) / 512")
    print("💡 Why? Neural networks perform better with normalized inputs")
    print()
    
    # Break down the normalization into sub-steps for clarity
    print("Sub-step 1a: Subtract offset (s - 512)")
    offset_removed = s - 512.0
    print(f"   {s.v} - 512 = {offset_removed.v}")
    print(f"   Derivative: d/ds[s - 512] = 1, so {s.d} - 0 = {offset_removed.d}")
    print()
    
    print("Sub-step 1b: Divide by scale factor ((s-512) / 512)")
    normalized = offset_removed / 512.0
    print(f"   {offset_removed.v} / 512 = {normalized.v}")
    print(f"   Derivative: d/ds[(s-512)/512] = 1/512 = {normalized.d}")
    print()
    
    print(f"✅ Normalization complete: {normalized}")
    print(f"   Interpretation: sensor value {s.v} → normalized value {normalized.v}")
    if normalized.v > 0:
        print(f"   This sensor reading is ABOVE the midpoint (512)")
    else:
        print(f"   This sensor reading is BELOW the midpoint (512)")
    print()
    
    # Step 2: Apply ReLU activation
    print("STEP 2: ReLU ACTIVATION")
    print("-" * 25)
    print("🎯 Goal: Remove negative values (keep only positive deviations)")
    print("📐 Formula: max(0, normalized_value)")
    print("💡 Why? Sometimes we only care about positive sensor deviations")
    print()
    
    activated = relu(normalized)
    print(f"✅ ReLU activation complete: {activated}")
    
    if activated.v > 0:
        print("   🟢 ReLU is ACTIVE: gradient flows through (derivative preserved)")
        print(f"   Gradient flow: {normalized.d} → {activated.d}")
    else:
        print("   🔴 ReLU is INACTIVE: gradient is blocked (derivative = 0)")
        print("   This means changes in sensor input won't affect the output")
    print()
    
    # Step 3: Scale by 0.5
    print("STEP 3: FINAL SCALING")
    print("-" * 25)
    print("🎯 Goal: Scale the activated value for neural network input")
    print("📐 Formula: activated_value × 0.5")
    print("💡 Why? Controls the input range to the neural network")
    print()
    
    scaled = activated * 0.5
    print(f"✅ Scaling complete: {scaled}")
    print(f"   Final feature value: {scaled.v}")
    print(f"   Final sensitivity: {scaled.d} (how much output changes per unit input change)")
    print()
    
    # Provide interpretation of the final result
    print("🔍 RESULT INTERPRETATION:")
    print("-" * 30)
    print(f"• Original sensor reading: {s.v}")
    print(f"• Processed feature value: {scaled.v}")
    print(f"• Sensitivity (derivative): {scaled.d}")
    print()
    print("What the sensitivity means:")
    if scaled.d > 0:
        print(f"  - A 1-unit increase in sensor reading causes a {scaled.d} increase in processed feature")
        print(f"  - A 10-unit sensor noise would cause ±{abs(scaled.d * 10):.6f} feature noise")
    else:
        print("  - Changes in sensor reading do NOT affect the processed feature")
        print("  - The system is insensitive to input changes in this region")
    print()
    
    return scaled

# Test the function with detailed walkthrough
print("🧪 TESTING THE PIPELINE")
print("=" * 60)
print()

# Test with sensor reading = 800 (above midpoint)
sensor_reading = 800.0
print(f"🎯 Test Case: Sensor reading = {sensor_reading}")
print(f"   This is {sensor_reading - 512} units above the midpoint (512)")
print(f"   Expected behavior: positive normalized value → ReLU active → derivative flows")
print()

# Create dual number for automatic differentiation
print("Creating dual number for AD computation...")
sensor = Dual(sensor_reading, 1.0)  # derivative = 1.0 for input variable
print(f"Input: {sensor} (derivative=1.0 means 'this is our input variable')")
print()

# Process the sensor data
result = tinyml_feature_processing(sensor)

print("=" * 60)
print("🎉 FINAL RESULTS")
print("=" * 60)
show("TinyML Feature Processing Results",
     ("Input sensor reading", sensor.v),
     ("Processed feature value", result.v),
     ("Sensitivity (∂output/∂input)", result.d),
     ("Sensitivity percentage", f"{result.d * 100:.6f}% per unit"))

print()
print("🎯 Key Insights:")
print(f"• The processing pipeline converts sensor value {sensor.v} to feature value {result.v}")
print(f"• The system has sensitivity {result.d:.10f} = 1/1024")
print(f"• This makes sense: normalization ÷512, then scaling ×0.5 = ×(1/1024)")
print(f"• The preprocessing is STABLE: small sensor changes cause tiny feature changes")

🧪 TESTING THE PIPELINE

🎯 Test Case: Sensor reading = 800.0
   This is 288.0 units above the midpoint (512)
   Expected behavior: positive normalized value → ReLU active → derivative flows

Creating dual number for AD computation...
Created dual number: value=800.0, derivative=1.0
Input: Dual(value=800.0, derivative=1.0) (derivative=1.0 means 'this is our input variable')

🔧 TINYML SENSOR PROCESSING PIPELINE
📊 Input sensor reading: Dual(value=800.0, derivative=1.0)
   Raw sensor value: 800.0
   Input derivative: 1.0 (1.0 means this is our input variable)

STEP 1: NORMALIZATION
-------------------------
🎯 Goal: Convert sensor range [0, 1024] to [-1, 1]
📐 Formula: (s - 512) / 512
💡 Why? Neural networks perform better with normalized inputs

Sub-step 1a: Subtract offset (s - 512)
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=288.0, derivative=1.0
Subtraction: Dual(value=800.0, derivative=1.0) - Dual(value=51

## 🧮 Mathematical Analysis of Our Results

### Understanding the Derivative Value

We got a derivative of **0.0009765625**. Let's understand what this means:

#### Mathematical Breakdown:
0.0009765625 = 1/1024

**Why exactly 1/1024?**

Let's trace through our operations:
1. **Normalization:** `(s - 512) / 512` → derivative = `1/512`
2. **ReLU:** For positive inputs → derivative = `1` (pass-through)
3. **Scaling:** `× 0.5` → derivative = `0.5`

**Combined:** `(1/512) × 1 × 0.5 = 1/1024`

### Engineering Interpretation:

#### Sensitivity Analysis:
- **1-unit sensor change** → **0.001 feature change**
- **10-unit sensor noise** → **0.01 feature noise**
- **100-unit sensor drift** → **0.1 feature change**

#### System Stability:
This low sensitivity indicates our preprocessing is **robust**:
- Small sensor noise won't significantly affect ML model input
- System is stable against environmental interference
- Gradients won't explode during training

#### Gradient Flow for Training:
If we're doing on-device learning:
- Gradients can flow back through our preprocessing
- The 1/1024 factor means preprocessing gradients will be small
- This helps prevent gradient explosion in the full system

In [7]:
    def numerical_gradient(f_scalar, x, epsilon=1e-6):
        """
        Compute numerical gradient using finite differences
        
        This is the "traditional" way of approximating derivatives before AD:
        f'(x) ≈ [f(x + ε) - f(x - ε)] / (2ε)
        
        This method:
        ✅ Works for any function
        ❌ Only gives approximations (not exact)
        ❌ Requires multiple function evaluations
        ❌ Sensitive to choice of epsilon
        ❌ Suffers from numerical precision issues
        
        Args:
            f_scalar: Function that takes a regular number and returns a regular number
            x: Point at which to compute gradient
            epsilon: Small step size for finite difference (too small = precision errors, too large = approximation errors)
        
        Returns:
            Numerical approximation of the derivative
        """
        print(f"🔢 Computing numerical gradient at x = {x}")
        print(f"   Using finite difference formula: [f(x+ε) - f(x-ε)] / (2ε)")
        print(f"   Step size ε = {epsilon}")
        
        # Compute function values at nearby points
        f_plus = f_scalar(x + epsilon)
        f_minus = f_scalar(x - epsilon)
        
        print(f"   f({x + epsilon}) = {f_plus}")
        print(f"   f({x - epsilon}) = {f_minus}")
        
        # Compute finite difference
        gradient = (f_plus - f_minus) / (2 * epsilon)
        print(f"   Gradient ≈ ({f_plus} - {f_minus}) / (2 × {epsilon}) = {gradient}")
        
        return gradient
    
    def tinyml_scalar(s):
        """
        Scalar version of our function for numerical gradient checking
        
        This function does the same computation as tinyml_feature_processing
        but uses only regular numbers (no dual numbers) so we can use it
        for numerical differentiation.
        """
        # Step 1: Normalize
        normalized = (float(s) - 512.0) / 512.0
        
        # Step 2: ReLU activation  
        activated = normalized if normalized > 0 else 0.0
        
        # Step 3: Scale
        scaled = activated * 0.5
        
        return scaled
    
    print("🔍 GRADIENT VERIFICATION")
    print("=" * 50)
    print()
    print("We'll now verify our automatic differentiation result using")
    print("the traditional numerical differentiation method.")
    print()
    print("This comparison will show:")
    print("✅ That our AD implementation is correct")
    print("✅ That AD gives more precise results than numerical methods")
    print("✅ That AD is more efficient (1 function call vs 2)")
    print()
    
    # Test point
    test_point = 800.0
    print(f"🎯 Test point: sensor reading = {test_point}")
    print()
    
    # Method 1: Numerical gradient (the old way)
    print("METHOD 1: NUMERICAL DIFFERENTIATION (Traditional)")
    print("-" * 55)
    numerical_grad = numerical_gradient(tinyml_scalar, test_point)
    print(f"   Result: {numerical_grad}")
    print()
    
    # Method 2: Automatic differentiation (the modern way)
    print("METHOD 2: AUTOMATIC DIFFERENTIATION (Modern)")
    print("-" * 55)
    print("🤖 Computing exact derivative using forward mode AD...")
    ad_input = Dual(test_point, 1.0)
    ad_result = tinyml_feature_processing(ad_input)
    print(f"   Result: {ad_result.d}")
    print()
    
    # Compare results
    print("🔍 DETAILED COMPARISON")
    print("=" * 30)
    error = abs(numerical_grad - ad_result.d)
    relative_error = error / abs(ad_result.d) if ad_result.d != 0 else float('inf')
    
    show("Gradient Comparison",
         ("Numerical (approximate)", f"{numerical_grad:.15f}"),
         ("Automatic Diff (exact)", f"{ad_result.d:.15f}"),
         ("Absolute error", f"{error:.2e}"),
         ("Relative error", f"{relative_error:.2e}"))
    
    print()
    if error < 1e-5:
        print("✅ VERIFICATION PASSED!")
        print("   AD gradient matches numerical gradient to high precision")
    else:
        print("❌ VERIFICATION FAILED!")
        print("   Significant difference detected - check implementation")
    
    print()
    print("🎯 WHY AUTOMATIC DIFFERENTIATION IS SUPERIOR:")
    print("-" * 50)
    print("1. 🎯 PRECISION:")
    print("   • AD gives EXACT derivatives (limited only by floating-point precision)")
    print("   • Numerical methods give approximations with truncation errors")
    print()
    print("2. ⚡ EFFICIENCY:")
    print("   • AD: 1 function evaluation (forward pass)")
    print("   • Numerical: 2+ function evaluations (for central differences)")
    print("   • For functions with n inputs, numerical needs 2n evaluations!")
    print()
    print("3. 🔒 STABILITY:")
    print("   • AD doesn't suffer from step-size sensitivity")
    print("   • Numerical methods can be unstable (too small ε → precision errors, too large ε → approximation errors)")
    print()
    print("4. 🚀 SCALABILITY:")
    print("   • AD scales to functions with millions of parameters")
    print("   • Numerical methods become prohibitively expensive")
    print()
    print("This is why every modern ML framework (TensorFlow, PyTorch, JAX) uses AD!")

🔍 GRADIENT VERIFICATION

We'll now verify our automatic differentiation result using
the traditional numerical differentiation method.

This comparison will show:
✅ That our AD implementation is correct
✅ That AD gives more precise results than numerical methods
✅ That AD is more efficient (1 function call vs 2)

🎯 Test point: sensor reading = 800.0

METHOD 1: NUMERICAL DIFFERENTIATION (Traditional)
-------------------------------------------------------
🔢 Computing numerical gradient at x = 800.0
   Using finite difference formula: [f(x+ε) - f(x-ε)] / (2ε)
   Step size ε = 1e-06
   f(800.000001) = 0.2812500009765625
   f(799.999999) = 0.2812499990234375
   Gradient ≈ (0.2812500009765625 - 0.2812499990234375) / (2 × 1e-06) = 0.0009765624975344167
   Result: 0.0009765624975344167

METHOD 2: AUTOMATIC DIFFERENTIATION (Modern)
-------------------------------------------------------
🤖 Computing exact derivative using forward mode AD...
Created dual number: value=800.0, derivative=1.0
🔧 TIN

## 🛠️ Engineering Applications and Real-World Context

### TinyML Use Cases Where This Matters

#### 1. Federated Learning on Edge Devices
**Scenario:** Smart home sensors learning personalized patterns
- **Challenge:** Each device must compute gradients locally
- **Our Solution:** Forward mode AD enables on-device gradient computation
- **Impact:** Privacy-preserving learning without sending raw data to cloud

#### 2. Adaptive Sensor Calibration
**Scenario:** Environmental sensors in harsh conditions
- **Challenge:** Sensor characteristics drift over time
- **Our Solution:** Derivatives tell us how to adjust preprocessing parameters
- **Impact:** Self-calibrating systems that maintain accuracy

#### 3. Real-Time Anomaly Detection
**Scenario:** Industrial equipment monitoring
- **Challenge:** Need to understand sensitivity to different sensor inputs
- **Our Solution:** Forward mode AD for real-time sensitivity analysis
- **Impact:** Early warning systems with explainable decisions

### Memory and Computational Analysis

#### Memory Requirements (Critical for Embedded Systems):
Traditional Neural Network Inference: N parameters × 4 bytes
- Forward Mode AD: N parameters × 8 bytes (2× overhead)
- Our preprocessing: 3 operations × 8 bytes = 24 bytes total

**For a 32KB embedded system:** Our preprocessing uses 0.075% of available memory!

#### Computational Requirements:
Original preprocessing: 3 operations (subtract, divide, multiply)
- Forward Mode AD: 6 operations total (2× overhead)
- Execution time: ~2× original (still real-time capable)

### Comparison with Alternatives

#### Forward Mode vs. Reverse Mode:
| Aspect | Forward Mode | Reverse Mode |
|--------|--------------|--------------|
| **Memory** | O(1) - constant | O(graph) - grows with computation |
| **Best for** | Few inputs, many outputs | Many inputs, few outputs |
| **TinyML fit** | ✅ Perfect | ❌ Memory hungry |
| **Real-time** | ✅ Predictable | ❌ Variable memory |


#### Why Forward Mode for Preprocessing:
- **Sensor processing:** Usually 1 input → 1 output (perfect for forward mode)
- **Memory constraints:** Embedded systems have KB, not GB
- **Real-time requirements:** Predictable execution time
- **Interpretability:** Easy to understand sensitivity to each input

In [8]:
print("🔬 ADVANCED ANALYSIS: Testing Edge Cases and System Behavior")
print("=" * 70)
print()

def analyze_sensitivity_across_range():
    """
    Analyze how our preprocessing behaves across the full sensor range.
    This helps us understand system behavior and potential issues.
    """
    print("📊 SENSITIVITY ANALYSIS ACROSS SENSOR RANGE")
    print("-" * 45)
    print()
    
    # Test different sensor readings
    test_points = [0, 256, 511, 512, 513, 768, 1024]
    
    print("Sensor | Normalized | ReLU | Scaled | Derivative | Active?")
    print("-------|------------|------|--------|------------|--------")
    
    for sensor_val in test_points:
        # Create dual number and process
        sensor = Dual(sensor_val, 1.0)
        
        # Process step by step (quietly)
        normalized = (sensor - 512.0) / 512.0
        activated = Dual(normalized.v, normalized.d) if normalized.v > 0 else Dual(0.0, 0.0)
        scaled = activated * 0.5
        
        # Determine if ReLU is active
        is_active = "Yes" if normalized.v > 0 else "No"
        
        print(f"{sensor_val:6d} | {normalized.v:10.3f} | {activated.v:4.3f} | {scaled.v:6.3f} | {scaled.d:10.6f} | {is_active}")
    
    print()
    print("🔍 Key Observations:")
    print("• Derivative is exactly 1/1024 = 0.000977 when ReLU is active")
    print("• Derivative is 0 when sensor reading ≤ 512 (ReLU inactive)")
    print("• There's a discontinuity at sensor = 512 (ReLU threshold)")
    print("• System is linear in the active region, zero in inactive region")

def test_extreme_cases():
    """Test behavior at extreme sensor values"""
    print()
    print("⚠️  EXTREME CASE TESTING")
    print("-" * 30)
    print()
    
    extreme_cases = [
        ("Very low sensor", 1.0),
        ("Just below threshold", 511.9),
        ("Just above threshold", 512.1),
        ("Very high sensor", 2000.0)
    ]
    
    for description, sensor_val in extreme_cases:
        print(f"🧪 {description}: {sensor_val}")
        sensor = Dual(sensor_val, 1.0)
        result = tinyml_scalar(sensor_val)  # Use scalar version for clean output
        
        # Compute derivative manually for this case
        if sensor_val > 512:
            derivative = 1.0 / 1024
            status = "Active"
        else:
            derivative = 0.0
            status = "Inactive"
        
        print(f"   Processed value: {result:.6f}")
        print(f"   Derivative: {derivative:.6f}")
        print(f"   ReLU status: {status}")
        print()

def demonstrate_gradient_flow():
    """Demonstrate how gradients flow through the preprocessing pipeline"""
    print("🔄 GRADIENT FLOW DEMONSTRATION")
    print("-" * 35)
    print()
    print("This shows how gradients would flow back through our preprocessing")
    print("during training in a larger ML system.")
    print()
    
    # Simulate a gradient coming back from the neural network
    upstream_gradient = 0.5  # Gradient from loss function
    sensor_val = 800.0
    
    print(f"📈 Scenario: Upstream gradient = {upstream_gradient}")
    print(f"   (This would come from the neural network during backpropagation)")
    print()
    
    # Compute forward pass with AD
    sensor = Dual(sensor_val, 1.0)
    result = tinyml_scalar(sensor_val)
    local_gradient = 1.0 / 1024  # Our preprocessing derivative
    
    # Apply chain rule
    total_gradient = upstream_gradient * local_gradient
    
    print(f"🔗 Chain rule application:")
    print(f"   Local gradient (preprocessing): {local_gradient:.6f}")
    print(f"   Upstream gradient (neural net): {upstream_gradient}")
    print(f"   Total gradient: {upstream_gradient} × {local_gradient:.6f} = {total_gradient:.6f}")
    print()
    print(f"📊 Interpretation:")
    print(f"   A small change in sensor reading would cause a {total_gradient:.6f} change")
    print(f"   in the final loss function. This guides the learning process!")

# Run all analyses
analyze_sensitivity_across_range()
test_extreme_cases()
demonstrate_gradient_flow()

print("=" * 70)
print("🎯 SUMMARY OF ADVANCED ANALYSIS")
print("=" * 70)
print("✅ System behaves predictably across the full sensor range")
print("✅ Gradients flow correctly through the preprocessing pipeline")
print("✅ ReLU creates a clear active/inactive boundary at sensor = 512")
print("✅ The preprocessing is robust and suitable for embedded deployment")

🔬 ADVANCED ANALYSIS: Testing Edge Cases and System Behavior

📊 SENSITIVITY ANALYSIS ACROSS SENSOR RANGE
---------------------------------------------

Sensor | Normalized | ReLU | Scaled | Derivative | Active?
-------|------------|------|--------|------------|--------
Created dual number: value=0.0, derivative=1.0
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=-512.0, derivative=1.0
Subtraction: Dual(value=0.0, derivative=1.0) - Dual(value=512.0, derivative=0.0) = Dual(value=-512.0, derivative=1.0)
Converting regular number 512.0 to Dual(512.0, 0.0)
Created dual number: value=512.0, derivative=0.0
Created dual number: value=-1.0, derivative=0.001953125
Division: Dual(value=-512.0, derivative=1.0) / Dual(value=512.0, derivative=0.0) = Dual(value=-1.0, derivative=0.001953125)
  Value: -512.0 / 512.0 = -1.0
  Derivative (Quotient Rule): (1.0 * 512.0 - -512.0 * 0.0) / 512.0² = 0.001953125
Created dual number: 